# Day 9 v3 — Silver: Invoice Metadata (Production — Job Parameters Only)

**Production notebook. Must be run via Databricks Job or ADF pipeline.**
**Interactive run will fail at Cell 2 — by design.**

---

## What this notebook does

Reads the Bronze invoice metadata Delta table (written by `day_6/04_bronze_blob_invoices_pdf.ipynb`),
applies type casting + data quality checks, and writes clean metadata to the Silver Delta table.

PDF content (amounts, line items) is not parsed here — that belongs in Gold.
Silver stores: `invoice_id`, `invoice_date`, `invoice_number`, `invoice_year/month/day`, `file_size_kb`, `bronze_path`.

---

## Parameters (injected by Databricks Job)

| Parameter | Required | Example | Notes |
|---|---|---|---|
| `load_type` | Yes | `incremental` | `full` or `incremental` |
| `ingestion_year` | Yes | `2026` | YYYY |
| `ingestion_month` | Yes | `06` | MM (zero-padded) |
| `ingestion_day` | No | `01` | DD — blank = all days in month |

---

## Data flow

```
Bronze: bronze-volume/invoices/_metadata/   (Delta)
         └─ invoice_id, year, month, day, file_size_kb, bronze_path

    filter by ingestion_year / ingestion_month / ingestion_day
    cast types → build invoice_date → extract invoice_number
    data quality → quarantine bad rows
    deduplicate on invoice_id → Delta MERGE

Silver: silver-volume/invoices/             (Delta — MERGE on invoice_id)
Silver: silver-volume/quarantine/invoices/  (Delta — rejected rows)
```

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Read parameters from Databricks Job ONLY (no defaults, no widget UI)

def _get_job_param(key):
    """Fail fast if parameter not injected by job."""
    try:
        value = dbutils.widgets.get(key).strip()
    except Exception:
        raise ValueError(
            f"Parameter '{key}' was not provided. "
            f"This notebook must be run via a Databricks Job — not interactively."
        )
    if not value:
        raise ValueError(f"Parameter '{key}' is empty. Set it in job task parameters.")
    return value

LOAD_TYPE       = _get_job_param("load_type")
INGESTION_YEAR  = _get_job_param("ingestion_year")
INGESTION_MONTH = _get_job_param("ingestion_month")

# Day is optional — blank = all days in given year/month
try:
    INGESTION_DAY = dbutils.widgets.get("ingestion_day").strip()
except Exception:
    INGESTION_DAY = ""

if LOAD_TYPE not in ("full", "incremental"):
    raise ValueError(f"load_type must be 'full' or 'incremental', got: {LOAD_TYPE!r}")

print(f"load_type       : {LOAD_TYPE}")
print(f"ingestion_year  : {INGESTION_YEAR}")
print(f"ingestion_month : {INGESTION_MONTH}")
print(f"ingestion_day   : {INGESTION_DAY or '(all days)'}")
print("Parameters validated.")

In [ ]:
# Cell 3 — Constants
BRONZE_METADATA_PATH = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/invoices/_metadata"
SILVER_PATH          = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/invoices"
QUARANTINE_PATH      = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/quarantine/invoices"

PIPELINE_NAME = "job_silver_blob_invoices_v3"
RUN_TS        = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze metadata : {BRONZE_METADATA_PATH}")
print(f"Silver path     : {SILVER_PATH}")
print(f"Quarantine path : {QUARANTINE_PATH}")
print(f"RUN_TS          : {RUN_TS}")

In [ ]:
# Cell 4 — Helper: check if DBFS/Volume path exists

def folder_exists_dbfs(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

print("folder_exists_dbfs() defined")

In [ ]:
# Cell 5 — Read Bronze invoice metadata Delta table with partition filter

if not folder_exists_dbfs(BRONZE_METADATA_PATH):
    raise Exception(
        f"Bronze metadata table not found at:\n  {BRONZE_METADATA_PATH}\n"
        f"Run day_6/04_bronze_blob_invoices_pdf.ipynb first."
    )

raw_df = spark.read.format("delta").load(BRONZE_METADATA_PATH)

if LOAD_TYPE == "incremental":
    raw_df = raw_df.filter(
        (F.col("year")  == INGESTION_YEAR) &
        (F.col("month") == INGESTION_MONTH)
    )
    if INGESTION_DAY:
        raw_df = raw_df.filter(F.col("day") == INGESTION_DAY)

BRONZE_ROW_COUNT = raw_df.count()
print(f"Bronze rows loaded : {BRONZE_ROW_COUNT}")
raw_df.show(3, truncate=False)

In [ ]:
# Cell 6 — Cast columns and build derived fields
# Bronze has string year/month/day — cast to integer, build invoice_date.
# Extract invoice_number (trailing numeric sequence from invoice_id).

cast_df = (
    raw_df
    .withColumn("invoice_id",   F.trim(F.col("invoice_id").cast("string")))
    .withColumn("invoice_year", F.col("year").cast("integer"))
    .withColumn("invoice_month",F.col("month").cast("integer"))
    .withColumn("invoice_day",  F.col("day").cast("integer"))
    .withColumn("file_size_kb", F.col("file_size_kb").cast("decimal(10,2)"))
    .withColumn("bronze_path",  F.trim(F.col("bronze_path").cast("string")))
    .drop("year", "month", "day")
    # invoice_date assembled from integer year/month/day
    .withColumn(
        "invoice_date",
        F.to_date(
            F.concat_ws("-",
                F.col("invoice_year").cast("string"),
                F.lpad(F.col("invoice_month").cast("string"), 2, "0"),
                F.lpad(F.col("invoice_day").cast("string"),   2, "0"),
            ), "yyyy-MM-dd"
        )
    )
    # invoice_number: trailing numeric part of invoice_id (INV-AU-2026-0002266 -> 2266)
    .withColumn(
        "invoice_number",
        F.regexp_extract(F.col("invoice_id"), r"(\d+)$", 1).cast("long")
    )
)

print("After cast and enrichment:")
cast_df.printSchema()
cast_df.show(3, truncate=False)

In [ ]:
# Cell 7 — Data quality: quarantine helper

def write_quarantine(df, reject_reason):
    if df.limit(1).count() == 0:
        return
    (
        df.withColumn("reject_reason", F.lit(reject_reason))
          .withColumn("reject_ts",     F.lit(RUN_TS).cast("timestamp"))
          .write.format("delta").mode("append").option("mergeSchema", "true")
          .save(QUARANTINE_PATH)
    )

print("write_quarantine() defined")

In [ ]:
# Cell 8 — Data quality: quarantine rows with null invoice_id

null_id_df = cast_df.filter(
    F.col("invoice_id").isNull() | (F.col("invoice_id") == "")
)
null_id_count = null_id_df.count()
write_quarantine(null_id_df, "null_invoice_id")

after_id_df = cast_df.filter(
    F.col("invoice_id").isNotNull() & (F.col("invoice_id") != "")
)

print(f"Quarantined (null_invoice_id) : {null_id_count}")
print(f"Remaining                     : {after_id_df.count()}")

In [ ]:
# Cell 9 — Data quality: quarantine rows with null invoice_date
# A null invoice_date means year/month/day parts were corrupted.

null_date_df = after_id_df.filter(F.col("invoice_date").isNull())
null_date_count = null_date_df.count()
write_quarantine(null_date_df, "null_invoice_date")

clean_df = after_id_df.filter(F.col("invoice_date").isNotNull())

print(f"Quarantined (null_invoice_date) : {null_date_count}")
print(f"Clean rows remaining            : {clean_df.count()}")

In [ ]:
# Cell 10 — Data quality: quarantine rows with negative file_size_kb
# A negative file size means the metadata was corrupted at Bronze.

neg_size_df = clean_df.filter(
    F.col("file_size_kb").isNotNull() & (F.col("file_size_kb") < 0)
)
neg_size_count = neg_size_df.count()
write_quarantine(neg_size_df, "negative_file_size_kb")

clean_df = clean_df.filter(
    F.col("file_size_kb").isNull() | (F.col("file_size_kb") >= 0)
)

print(f"Quarantined (negative_file_size_kb) : {neg_size_count}")
print(f"Clean rows remaining                : {clean_df.count()}")

In [ ]:
# Cell 11 — Add Silver audit columns

audited_df = (
    clean_df
    .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
    .withColumn("silver_load_type",   F.lit(LOAD_TYPE))
    .withColumn("silver_pipeline",    F.lit(PIPELINE_NAME))
)

print(f"Rows with audit columns: {audited_df.count()}")

In [ ]:
# Cell 12 — Deduplicate on invoice_id (latest silver_ingested_at wins)

window = Window.partitionBy("invoice_id").orderBy(F.col("silver_ingested_at").desc())
deduped_df = (
    audited_df
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

SILVER_ROW_COUNT = deduped_df.count()
print(f"After dedup: {SILVER_ROW_COUNT} rows")

In [ ]:
# Cell 13 — Write to Silver Delta (full overwrite or MERGE)

if LOAD_TYPE == "full" or not folder_exists_dbfs(SILVER_PATH):
    (
        deduped_df.write.format("delta")
        .mode("overwrite").option("overwriteSchema", "true")
        .save(SILVER_PATH)
    )
    print(f"Full overwrite complete — {SILVER_ROW_COUNT} rows written")
else:
    delta_table = DeltaTable.forPath(spark, SILVER_PATH)
    (
        delta_table.alias("target")
        .merge(
            deduped_df.alias("source"),
            "target.invoice_id = source.invoice_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"MERGE complete — {SILVER_ROW_COUNT} rows upserted")

In [ ]:
# Cell 14 — Run summary

silver_df = spark.read.format("delta").load(SILVER_PATH)
silver_total = silver_df.count()

print("=" * 65)
print("SILVER INVOICE METADATA v3 — RUN SUMMARY")
print("=" * 65)
print(f"  run_timestamp   : {RUN_TS}")
print(f"  load_type       : {LOAD_TYPE}")
print(f"  partition       : {INGESTION_YEAR}/{INGESTION_MONTH}/{INGESTION_DAY or '*'}")
print(f"  bronze_rows     : {BRONZE_ROW_COUNT}")
print(f"  silver_rows     : {SILVER_ROW_COUNT}  (this run)")
print(f"  silver_total    : {silver_total}  (all-time)")
print(f"  quarantined     : {BRONZE_ROW_COUNT - SILVER_ROW_COUNT}")
print("=" * 65)

print("\nInvoices per month (Silver total):")
silver_df.groupBy("invoice_year", "invoice_month").count() \
         .orderBy("invoice_year", "invoice_month").show()